# Cardio Catch Disease — 01. Data Understanding

> **Hands-On ML principle (Ch. 2):** Take a quick look at the data structure with `head()`, `info()`, `describe()`. Create the test set and **lock it away** before any exploration to avoid data snooping bias.

---

## 0. Setup

In [ ]:
from pathlib import Path
import sys, os, warnings

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib

matplotlib.rcParams["figure.dpi"] = 120
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 20)

FIGURES = PROJECT_ROOT / "reports" / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)
print(f"Project root: {PROJECT_ROOT}")

## 1. Load Raw Data

In [ ]:
from cardio_catch_disease.data import load_training_frame
from cardio_catch_disease.config import load_config

config = load_config(PROJECT_ROOT / "configs" / "project.toml")
df = load_training_frame(config)

print(f"Dataset shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
df.head()

## 2. Create Stratified Train / Test Split — Set Test Aside

> **Book warning (Ch. 2):** "Before you look at the data any further, you need to create a test set, put it aside, and never look at it."

We use **stratified sampling** on `cardio` to preserve the class ratio in both splits.

In [ ]:
from sklearn.model_selection import train_test_split

train_set, test_set = train_test_split(
    df, test_size=0.20, random_state=42, stratify=df["cardio"]
)

print(f"Training set : {train_set.shape[0]:,} rows ({len(train_set)/len(df):.0%})")
print(f"Test set     : {test_set.shape[0]:,} rows ({len(test_set)/len(df):.0%})")
print()

for name, split in [("train", train_set), ("test", test_set)]:
    dist = split["cardio"].value_counts(normalize=True)
    print(f"Cardio distribution ({name}): No Disease={dist[0]:.1%}  Disease={dist[1]:.1%}")

# From here on we explore ONLY the training set
df_train = train_set.copy()
print("\n✓ Test set locked — will only be used in Notebook 04 for final evaluation.")

## 3. Data Schema — info()

In [ ]:
df_train.info()

**Key observations:**
- **No missing values** — all 56,000 rows are complete
- `age` is stored in **days** (will be converted to years as a feature)
- All features are numeric (int64 / float64) — no raw text encoding needed
- `gender`, `cholesterol`, `gluc`, `smoke`, `alco`, `active`, `cardio` are ordinal/binary codes

## 4. Statistical Summary — describe()

In [ ]:
df_train.describe().round(2)

**Red flags from describe():**

| Feature | Issue | Action |
|---|---|---|
| `ap_hi` min = −150 | Physiologically impossible | Filter out (< 70 or > 250) |
| `ap_hi` max = 16,020 | Clearly a data entry error | Filter out |
| `ap_lo` min = −70 | Physiologically impossible | Filter out (< 40 or > 150) |
| `ap_hi < ap_lo` | Diastolic cannot exceed systolic | Filter out |
| `age` in days (mean ≈ 19,000) | Needs conversion to years | Create `age_years` feature |

## 5. Missing Values Audit

In [ ]:
missing = df_train.isnull().sum()
if missing.sum() == 0:
    print("✓ No missing values. Dataset is complete.")
else:
    display(pd.DataFrame({"count": missing[missing > 0], "pct": (missing[missing > 0]/len(df_train)*100).round(2)}))

## 6. Feature Distributions — hist()

In [ ]:
# Following the book's approach: plot a histogram for each numerical attribute
plot_df = df_train.copy()
plot_df["age_years"] = plot_df["age"] / 365.25  # more interpretable
plot_df["ap_hi_clipped"] = plot_df["ap_hi"].clip(50, 260)
plot_df["ap_lo_clipped"] = plot_df["ap_lo"].clip(30, 180)

cols_to_plot = ["age_years", "height", "weight", "ap_hi_clipped", "ap_lo_clipped",
                "cholesterol", "gluc", "smoke", "alco", "active", "gender", "cardio"]
labels = ["Age (years)", "Height (cm)", "Weight (kg)", "Systolic BP (clipped)",
          "Diastolic BP (clipped)", "Cholesterol", "Glucose", "Smoker",
          "Alcohol", "Active", "Gender", "Cardio (target)"]

fig, axes = plt.subplots(3, 4, figsize=(15, 10))
colors = plt.cm.tab10.colors
for ax, col, label, color in zip(axes.flatten(), cols_to_plot, labels, colors):
    plot_df[col].hist(bins=40, ax=ax, color=color, edgecolor="white", linewidth=0.5)
    ax.set_title(label, fontsize=9, fontweight="bold")
    ax.set_xlabel("Value", fontsize=8)

plt.suptitle("Feature Distributions — Training Set (N=56,000)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(FIGURES / "01_feature_distributions.png", bbox_inches="tight")
plt.show()

**Insights from histograms:**

1. **Age:** Concentrated between 40–60 years. Roughly symmetric.
2. **Height / Weight:** Near-normal distributions with some outliers
3. **Blood pressure:** Heavy right tail — outliers confirmed, need filtering
4. **Cholesterol / Glucose:** Majority at level 1 (normal), minority at elevated levels
5. **Smoke / Alco:** Strongly imbalanced — few patients in these categories
6. **Target (cardio):** Near **50/50 balance** — no need for resampling

## 7. Target Class Balance

In [ ]:
counts = df_train["cardio"].value_counts()
pcts   = df_train["cardio"].value_counts(normalize=True)

print("Target distribution in training set:")
print(pd.DataFrame({"Count": counts, "Share": pcts.map("{:.1%}".format)})
      .rename(index={0: "No Disease (0)", 1: "Disease (1)"}))

fig, ax = plt.subplots(figsize=(5, 4))
counts.rename({0: "No Disease", 1: "Disease"}).plot.bar(
    ax=ax, color=["#2ecc71", "#e74c3c"], edgecolor="white", rot=0)
ax.set_title("Target Class Balance", fontweight="bold")
ax.set_ylabel("Patient count")
for bar in ax.patches:
    ax.annotate(f"{int(bar.get_height()):,}",
                (bar.get_x() + bar.get_width()/2, bar.get_height()),
                ha="center", va="bottom", fontsize=11)
plt.tight_layout()
plt.savefig(FIGURES / "01_target_balance.png", bbox_inches="tight")
plt.show()

## 8. Categorical Feature Frequencies

In [ ]:
cat_meta = {
    "gender":      {1: "Female", 2: "Male"},
    "cholesterol": {1: "Normal", 2: "Above Normal", 3: "Well Above"},
    "gluc":        {1: "Normal", 2: "Above Normal", 3: "Well Above"},
    "smoke":       {0: "Non-Smoker", 1: "Smoker"},
    "alco":        {0: "No Alcohol",  1: "Drinks"},
    "active":      {0: "Inactive",    1: "Active"},
}

fig, axes = plt.subplots(2, 3, figsize=(13, 7))
for ax, (col, lmap) in zip(axes.flatten(), cat_meta.items()):
    vc = df_train[col].value_counts().sort_index()
    vc.index = [lmap.get(k, str(k)) for k in vc.index]
    vc.plot.bar(ax=ax, color="#3498db", edgecolor="white", rot=25)
    ax.set_title(col.capitalize(), fontweight="bold")
    ax.set_ylabel("Count")
    for bar in ax.patches:
        ax.annotate(f"{int(bar.get_height()):,}",
                    (bar.get_x() + bar.get_width()/2, bar.get_height()),
                    ha="center", va="bottom", fontsize=8)

plt.suptitle("Categorical Feature Frequencies", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(FIGURES / "01_categorical_frequencies.png", bbox_inches="tight")
plt.show()

## 9. Blood Pressure Outlier Quantification

In [ ]:
bp_mask = (
    (df_train["ap_hi"] < 70)  | (df_train["ap_hi"] > 250) |
    (df_train["ap_lo"] < 40)  | (df_train["ap_lo"] > 150) |
    (df_train["ap_hi"] <= df_train["ap_lo"])  # systolic must exceed diastolic
)
n_outliers = bp_mask.sum()
print(f"Blood pressure outliers: {n_outliers:,} ({n_outliers/len(df_train):.1%} of training rows)")
print("These will be removed during feature engineering (Notebook 03).")

# Quick visualisation
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, col, valid_range, color in [
    (axes[0], "ap_hi",  (70, 250),  "#3498db"),
    (axes[1], "ap_lo",  (40, 150),  "#e74c3c"),
]:
    df_train[col].clip(-50, 300).hist(bins=60, ax=ax, color=color, edgecolor="white")
    ax.axvline(valid_range[0], color="red",  linestyle="--", label=f"Min ({valid_range[0]})")
    ax.axvline(valid_range[1], color="navy", linestyle="--", label=f"Max ({valid_range[1]})")
    ax.set_title(f"{col} — Raw Distribution", fontweight="bold")
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(FIGURES / "01_bp_outliers.png", bbox_inches="tight")
plt.show()

## 10. Summary

| Finding | Action (Notebook) |
|---|---|
| No missing values | Pipeline still includes imputer for robustness |
| Age in days | Create `age_years` feature (03) |
| Blood pressure outliers (~3 %) | Filter during data cleaning (03) |
| Near-balanced target (50/50) | `class_weight='balanced'` as safety net (04) |
| No raw text features | Standard sklearn pipeline works out of the box |
| BMI not pre-calculated | Create from height + weight (03) |

**Next → Notebook 02: Exploratory Analysis**